# Validating SIMBA-POP: three ways to drive the engine, and a naive baseline

`SimbaPopEngine` can be fed three different amounts of real information. This notebook runs all
three from the same farm inventory and scores the simulated weekly harvest (`Ht`) against what was
actually cut (`cut`) -- plus a naive persistence baseline.

- **Mode A -- purely mechanistic.** Starts from the farm's realistic initial population -- an
  inventory of an established farm (for the default `data/synthetic`, its field census) or, for a
  new farm, its staggered planting schedule (see
  [notebook 02](02_initial_population_structure_and_growth_limits.ipynb) and
  `data/synthetic_new_plantation`) -- and is then driven only by temperature, with Tixier et al.
  (2004)'s default parameters and no field records after the start. This recreates the paper's own
  Fig 2 (harvest dynamics), Fig 5 (weekly scatter), Fig 6 (harvesting-peak timing) and Table 4
  (peak-timing errors).
- **Mode B -- ribbon production only.** The farm's weekly ribbon-tag bagging drives the
  post-flowering chains; the pre-flowering chain's new followers still come from the mechanistic
  Eq 3 prediction (Tixier's sucker-selection curve), because `sucker_selection=None`.
- **Mode C -- ribbon production + sucker selection.** The same bagging, plus the farm's weekly
  desuckering records driving the pre-flowering chain. Run only when the data directory has a
  `sucker_selection.csv`.
- **Naive baseline** -- hold the most recently observed weekly cut flat over the forecast horizon.
  A mode that can't beat this adds nothing a farm manager couldn't do by eye.

Every mode uses the default per-cohort rate basis (`general.rate_basis = "cohort"`): every plant
flowers once, is harvested once and keeps exactly one follower, so the farm's number of mats stays
constant without any carrying capacity.

The data directory defaults to `data/synthetic` (an established 165-ha synthetic farm) and can be
switched with the `SIMBA_POP_DATA_DIR` environment variable -- e.g. `data/synthetic_new_plantation`,
or a real farm directory without desuckering records (Modes A/B only). The synthetic farms' records
were generated by this same model (Tixier biology) combined with the farm's own desuckering
practice (Guarin & Ochoa 2012), so on them a near-perfect fit
is a consistency check of the whole pipeline, not an independent validation of Tixier's parameters.

In [1]:
from pathlib import Path

# This notebook lives in docs/examples/: run it from there so data/ resolves
# (py_simba_pop itself comes from `pip install py-simba-pop`).

In [2]:
import os
from collections import Counter

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.optimize import linear_sum_assignment
from scipy.signal import find_peaks

from py_simba_pop import (
    UNASSIGNED_TAG,
    ParameterSet,
    PopulationInventoryReader,
    RibbonProductionData,
    RibbonProductionReader,
    SimbaPopEngine,
    SuckerSelectionData,
    SuckerSelectionReader,
    WeatherProvider,
)

## 1. Data directory and shared constants

In [3]:
DATA_DIR = Path(os.environ.get("SIMBA_POP_DATA_DIR", "data/synthetic"))  # or data/synthetic_new_plantation, data/<farm>
FORECAST_HORIZON = 12  # weeks ahead, main rolling sweep (Sections 12-18)
LONG_HORIZON, LONG_STRIDE = 26, 4  # long-horizon Mode B vs. Mode C sweep (Section 19)
params = ParameterSet.from_tixier_defaults()  # Tixier et al. (2004) defaults throughout this notebook

print(f"DATA_DIR = {DATA_DIR.as_posix()}, rate_basis = {params.general['rate_basis']!r}")
print(params)

DATA_DIR = data/synthetic, rate_basis = 'cohort'
ParameterSet(general={'base_temperature_c': 14.0, 'week_length_days': 7, 'population_epsilon': 0.5, 'max_age_weeks_vegetative': 150, 'max_age_weeks_reproductive': 20, 'rate_basis': 'cohort'}, flowering={'threshold_dd': 2350.0, 'amplitude': 0.12, 'peak_weeks': 6.35, 'spread': 0.47}, sucker_selection={'threshold_dd': 2180.0, 'amplitude': 1.0, 'peak_weeks': 3.25, 'spread': 0.31, 'carrying_capacity': None}, harvest={'threshold_dd': 900.0, 'hazard_weeks': [0.2, 0.625, 1.0]}, mortality_replacement={'death_rate_vegetative': 0.0, 'death_rate_reproductive': 0.0, 'replacement_rate': 0.0}, source=('tixier_defaults',))


## 2. Input files

Every file this notebook reads is a canonical csv in the schema of the package's readers:
`weather.csv`, `production.csv`, the optional `sucker_selection.csv`, and `initial_population.csv`
(a field census, a new farm's planting schedule, or a reconstruction from the production records
with py-simba-pop-tools). Converting a farm's own exports into these files is the job of a separate,
farm-specific adapter.

In [4]:
INPUT_FILES = {
    "weather.csv": "required -- defines the week grid",
    "production.csv": "required -- ground truth (cut); drives Modes B/C (bagged)",
    "initial_population.csv": "required -- every mode starts from it",
    "sucker_selection.csv": "optional -- Mode C only",
}
pd.DataFrame(
    [
        {
            "file": name,
            "role": role,
            "present": (DATA_DIR / name).exists(),
            "rows": len(pd.read_csv(DATA_DIR / name)) if (DATA_DIR / name).exists() else 0,
        }
        for name, role in INPUT_FILES.items()
    ]
)

,file,role,present,rows
0,weather.csv,required -- defines the week grid,True,3640
1,production.csv,required -- ground truth (cut); drives Modes B...,True,2879
2,initial_population.csv,required -- every mode starts from it,True,92
3,sucker_selection.csv,optional -- Mode C only,True,359


## 3. Weather and the scoring window

`weather.csv` (daily mean temperature) defines the week grid: `week_index` 0 is the week starting
on the first weather date. The initial population is a snapshot taken at the start of one week;
its `snapshot_date` column fixes `START_WEEK_INDEX`, the first simulated week (the inventory
reader ignores that column, so it is read from the raw csv). `LAST_USABLE_WEEK` is the last week
covered by both weather and production records, and every run simulates
`START_WEEK_INDEX..LAST_USABLE_WEEK` inclusive -- the scoring window used throughout.

In [5]:
weather = WeatherProvider.from_file_and_params(DATA_DIR / "weather.csv", params)

prod_result = RibbonProductionReader.read(DATA_DIR / "production.csv")
production_raw = prod_result.data
production = production_raw.assign(week_index=production_raw["date"].map(weather.week_index_for_date))
FIRST_PRODUCTION_WEEK = int(production["week_index"].min())
LAST_PRODUCTION_WEEK = int(production["week_index"].max())

snapshot_date = pd.to_datetime(pd.read_csv(DATA_DIR / "initial_population.csv")["snapshot_date"].iloc[0]).date()
START_WEEK_INDEX = weather.week_index_for_date(snapshot_date)
LAST_USABLE_WEEK = min(weather.max_week_index, LAST_PRODUCTION_WEEK)
N_WEEKS = LAST_USABLE_WEEK - START_WEEK_INDEX + 1

print(f"weather covers week_index {weather.min_week_index}..{weather.max_week_index} (weeks starting "
      f"{weather.calendar_date_for_week(weather.min_week_index)} .. {weather.calendar_date_for_week(weather.max_week_index)})")
print(f"production records cover week_index {FIRST_PRODUCTION_WEEK}..{LAST_PRODUCTION_WEEK}")
print(f"initial population snapshot_date {snapshot_date} -> START_WEEK_INDEX = {START_WEEK_INDEX}")
print(f"LAST_USABLE_WEEK = {LAST_USABLE_WEEK} ({weather.calendar_date_for_week(LAST_USABLE_WEEK)}); "
      f"every run simulates {N_WEEKS} weeks, {START_WEEK_INDEX}..{LAST_USABLE_WEEK}")

weather covers week_index 0..519 (weeks starting 2016-01-04 .. 2025-12-15)
production records cover week_index 156..516
initial population snapshot_date 2018-12-31 -> START_WEEK_INDEX = 156
LAST_USABLE_WEEK = 516 (2025-11-24); every run simulates 361 weeks, 156..516


## 4. Standardized readers

`RibbonProductionData` wraps the validated production table. `SuckerSelectionData` is built only
when `sucker_selection.csv` exists -- otherwise `sucker_selection=None`, Mode C is left out of
`MODES`, and every Mode C cell below is skipped. Notices are printed with a small helper rather
than re-emitted as Python warnings.

In [6]:
def print_notices(notices, limit: int = 5) -> None:
    """A count per (kind, subject), then up to `limit` individual notices per subject."""
    counts = Counter((n.kind.value, n.subject) for n in notices)
    if not counts:
        print("  (no notices)")
        return
    for (kind, subject), count in sorted(counts.items()):
        print(f"  {count:>4} x [{kind}] {subject}")
    shown = Counter()
    for n in notices:
        if shown[n.subject] < limit:
            shown[n.subject] += 1
            where = "" if n.week_index is None else f" week_index={n.week_index}"
            where += "" if n.tag is None else f" tag={n.tag}"
            print(f"       [{n.kind.value}] {n.subject}{where}: {n.message}")


ribbon_production = RibbonProductionData(production_raw, weather)
print(f"ribbon tags: {ribbon_production.tags()}")
print("production reader notices:")
print_notices(prod_result.notices)

sucker_selection, sel_result = None, None
if (DATA_DIR / "sucker_selection.csv").exists():
    sel_result = SuckerSelectionReader.read(DATA_DIR / "sucker_selection.csv")
    print("sucker-selection reader notices:")
    print_notices(sel_result.notices)
    if len(sel_result.data):
        sucker_selection = SuckerSelectionData(sel_result.data, weather)

MODES = ["Mode A", "Mode B"] + (["Mode C"] if sucker_selection is not None else [])
print(f"sucker_selection: {'present' if sucker_selection is not None else 'None'} -> MODES = {MODES}")

ribbon tags: ('Amarillo', 'Azul', 'Blanco', 'Café', 'Morado', 'Negro', 'Rojo', 'Verde')
production reader notices:
     1 x [feature_disabled] processed
       [feature_disabled] processed: 'processed' column not present in WindowsPath('data/synthetic/production.csv'); this is a non-paper extension with no Tixier equation to estimate it from, so it will simply not be modeled (treated as absent, not zero).
sucker-selection reader notices:
  (no notices)
sucker_selection: present -> MODES = ['Mode A', 'Mode B', 'Mode C']


In [7]:
initial_population = PopulationInventoryReader.read(DATA_DIR / "initial_population.csv").data
display(
    initial_population.groupby("chain").agg(
        cohorts=("population", "size"),
        plants=("population", "sum"),
        min_age_weeks=("age_weeks", "min"),
        max_age_weeks=("age_weeks", "max"),
    )
)
total_population_at_start = initial_population["population"].sum()
initial_post_stock = initial_population.loc[initial_population["chain"] == "post_flowering", "population"].sum()
print(f"standing plants at the start of week {START_WEEK_INDEX}: {total_population_at_start:,.0f} "
      f"(of which {initial_post_stock:,.0f} already flowered, bunch on the plant)")

,cohorts,plants,min_age_weeks,max_age_weeks
chain,,,,
post_flowering,14,94820.0,1,14
pre_flowering,78,322476.0,1,78


standing plants at the start of week 156: 417,296 (of which 94,820 already flowered, bunch on the plant)


## 5. Observed weekly harvest and bagging coverage

Pooled `cut` across every tag, per week, is the ground truth every mode is scored against; pooled
`bagged` is what drives Modes B/C. Real records are not always complete, so the weekly bagged
total is scanned for zero or unrecorded weeks, starting at the first production week (before it
there is simply nothing recorded yet).

In [8]:
SCORED_WEEKS = pd.RangeIndex(START_WEEK_INDEX, LAST_USABLE_WEEK + 1, name="week_index")
week_dates = pd.Series([weather.calendar_date_for_week(w) for w in SCORED_WEEKS], index=SCORED_WEEKS)

recorded_weeks = pd.RangeIndex(FIRST_PRODUCTION_WEEK, LAST_USABLE_WEEK + 1, name="week_index")
weekly = production.groupby("week_index")[["cut", "bagged"]].sum()
observed_cut_all = weekly["cut"].reindex(recorded_weeks, fill_value=0.0)  # from the first production week
observed_cut = weekly["cut"].reindex(SCORED_WEEKS, fill_value=0.0)  # the scoring window
observed_bagged = weekly["bagged"].reindex(SCORED_WEEKS, fill_value=0.0)

zero_runs, run_start = [], None
for w, value in weekly["bagged"].reindex(recorded_weeks, fill_value=0.0).items():
    if value == 0 and run_start is None:
        run_start = w
    elif value != 0 and run_start is not None:
        zero_runs.append((run_start, w - 1))
        run_start = None
if run_start is not None:
    zero_runs.append((run_start, LAST_USABLE_WEEK))

print(f"scoring window: mean {observed_cut.mean():,.0f} bunches cut/week, {observed_bagged.mean():,.0f} bagged/week")
print(f"zero/unrecorded bagging weeks since the first production week {FIRST_PRODUCTION_WEEK}: "
      f"{sum(b - a + 1 for a, b in zero_runs)} of {len(recorded_weeks)}; "
      f"longest run {max((b - a + 1 for a, b in zero_runs), default=0)} weeks; runs {zero_runs}")
production.head()

scoring window: mean 8,447 bunches cut/week, 8,664 bagged/week
zero/unrecorded bagging weeks since the first production week 156: 0 of 361; longest run 0 weeks; runs []


,tag,date,bagged,cut,age_at_cut,field_losses,rejected,processed,week_index
0,Amarillo,2018-12-31,NaN,NaN,NaN,24.0,NaN,NaN,156
1,Azul,2018-12-31,NaN,1425.0,12.0,20.0,33.0,NaN,156
2,Blanco,2018-12-31,NaN,NaN,NaN,29.0,NaN,NaN,156
3,Café,2018-12-31,NaN,NaN,NaN,4.0,NaN,NaN,156
4,Morado,2018-12-31,NaN,NaN,NaN,29.0,NaN,NaN,156


A multi-week stretch of zero recorded bagging while cutting continues is a gap in the source
records, not a model bug: Modes B/C then have nothing new entering the post-flowering chains, so
any 12-week forecast whose cutoff falls during or just after such a gap under-predicts harvest by
construction (a bunch is cut roughly 12 weeks after it is bagged). Check this scan before
attributing a Mode B/C error to the model.

## 6. One engine, three ways to feed it

`build_engine(mode)` is the only way this notebook constructs a driven engine: the modes differ
only in which optional real-data arguments of the same `SimbaPopEngine` constructor are filled in
-- never a different code path.

In [9]:
def build_engine(mode: str) -> SimbaPopEngine:
    real_data = {}
    if mode in ("Mode B", "Mode C"):
        real_data = {"ribbon_production": ribbon_production, "tags": list(ribbon_production.tags())}
    if mode == "Mode C":
        if sucker_selection is None:
            raise ValueError("Mode C needs sucker_selection.csv in DATA_DIR")
        real_data["sucker_selection"] = sucker_selection
    return SimbaPopEngine(
        parameters=params,
        weather=weather,
        initial_population=initial_population,
        start_week_index=START_WEEK_INDEX,
        **real_data,
    )

## 7. Mode A -- purely mechanistic from the initial population

Only `initial_population` and temperature: no `ribbon_production`, no `sucker_selection`, no field
records after the start. Every flow follows Tixier's default curves, and new flowering routes to
the reserved `UNASSIGNED_TAG` chain (no ribbon colour is known).

In [10]:
mode_a_engine = build_engine("Mode A")
mode_a_output = mode_a_engine.run(N_WEEKS)
print(f"Mode A simulated week_index {mode_a_output['week_index'].min()}..{mode_a_output['week_index'].max()}; notices:")
print_notices(mode_a_engine.notices)
mode_a_output.tail()

Mode A simulated week_index 156..516; notices:
     1 x [fallback_estimate] ribbon_production
     1 x [fallback_estimate] sucker_selection
       [fallback_estimate] ribbon_production: no ribbon-tag production data provided -- every week's flowering output will route to the reserved '__unassigned__' chain, and harvest follows the Tixier hazard curve exactly as in a fully mechanistic run. Provide ribbon_production= to drive specific tags from real bagged counts.
       [fallback_estimate] sucker_selection: no sucker-selection data provided -- the pre-flowering chain's weekly inflow will use the model's own mechanistic Eq 3 pooled-sucker prediction for the entire run. Provide sucker_selection= to use real observed desuckering counts instead.


,week_index,date,Nt,Nt_pre,Nt_post,Ht,FLt,St,sucker_output_pre,sucker_output_post,Dt,sucker_input_used,flowering_routed_to_unassigned,notice_count,pruned_population_this_week
356,512,2025-10-27,425211.131663,317885.381608,107325.750055,9172.543712,8060.448599,9290.290004,9234.572336,55.717668,0.0,9290.290004,8060.448599,0,0.439624
357,513,2025-11-03,422859.882864,316661.838801,106198.044062,8920.046729,7792.340736,6569.237049,6521.767351,47.469698,0.0,6569.237049,7792.340736,0,0.439119
358,514,2025-11-10,420796.766132,315763.197009,105033.569123,8705.197668,7540.722729,6642.560430,6607.540747,35.019683,0.0,6642.560430,7540.722729,0,0.479494
359,515,2025-11-17,422433.223816,316737.722669,105695.501147,6840.412988,7502.345013,8477.326605,8449.222926,28.103679,0.0,8477.326605,7502.345013,0,0.455932
360,516,2025-11-24,428075.229726,317878.910988,110196.318738,3150.440369,7651.257960,8792.905964,8763.307655,29.598308,0.0,8792.905964,7651.257960,0,0.459685


### Mode A stays at the farm's scale

Under the per-cohort rate basis every plant flowers once, is harvested once and leaves exactly one
follower, so the number of mats (production units) is fixed and the standing population
`Nt = Nt_pre + Nt_post` stays at the farm's own scale -- an established farm hovers around its
starting inventory (a mother plus, most of the time, its follower), while a new plantation rises
from one plant per mat as each mat adds its follower, then levels off. No `carrying_capacity` is
needed. `sucker_output_pre`/`sucker_output_post` split the selected followers by whether their
mother had already flowered (nulliparous/parous). Under the legacy pre-0.3 reading
(`params.override("general", {"rate_basis": "current_pool"})`), where ~35% of plants never flower
and each yields ~2.6 followers, the same kind of run grew ~5x per year.

In [11]:
fig = go.Figure()
for column, name, color in (
    ("Nt", "Nt (all standing plants)", "#1565C0"),
    ("Nt_pre", "Nt_pre (not yet flowered)", "#2E7D32"),
    ("Nt_post", "Nt_post (flowered, bunch on the plant)", "#C62828"),
):
    fig.add_trace(go.Scatter(x=mode_a_output["date"], y=mode_a_output[column], name=name, line=dict(color=color)))
fig.update_layout(
    title="Mode A: standing population, temperature-driven only",
    xaxis_title="Date", yaxis_title="Plants", template="plotly_white", legend=dict(orientation="h", y=1.15),
)
fig.show()

nt = mode_a_output["Nt"]
years = N_WEEKS / 52.18
print(f"Mode A Nt: {nt.iloc[0]:,.0f} after the first week, {nt.iloc[-1]:,.0f} after {N_WEEKS} weeks "
      f"(range {nt.min():,.0f}..{nt.max():,.0f}; initial inventory {total_population_at_start:,.0f})")
print(f"Mode A harvest: {mode_a_output['Ht'].sum() / years:,.0f} bunches/yr simulated vs. "
      f"{observed_cut.sum() / years:,.0f} bunches/yr observed cut")
print(f"Mode A followers selected: {mode_a_output['sucker_output_pre'].mean():,.0f}/week for not-yet-flowered mothers, "
      f"{mode_a_output['sucker_output_post'].mean():,.0f}/week for already-flowered mothers")

Mode A Nt: 419,575 after the first week, 428,075 after 361 weeks (range 408,279..443,383; initial inventory 417,296)
Mode A harvest: 445,026 bunches/yr simulated vs. 440,751 bunches/yr observed cut
Mode A followers selected: 8,498/week for not-yet-flowered mothers, 61/week for already-flowered mothers


## 8. Mode B -- ribbon production only

Real per-tag bagging seeds every post-flowering chain; `sucker_selection=None`, so the
pre-flowering chain's new followers come from the mechanistic Eq 3 prediction every week (one
construction-time notice says so). Harvest `Ht` is always the mechanistic hazard curve -- the real
`cut` is never fed back into the state.

In [12]:
mode_b_engine = build_engine("Mode B")
mode_b_output = mode_b_engine.run(N_WEEKS)
print("Mode B notices:")
print_notices(mode_b_engine.notices)
mode_b_output.tail()

Mode B notices:
     1 x [fallback_estimate] sucker_selection
     4 x [feature_disabled] field_losses
       [fallback_estimate] sucker_selection: no sucker-selection data provided -- the pre-flowering chain's weekly inflow will use the model's own mechanistic Eq 3 pooled-sucker prediction for the entire run. Provide sucker_selection= to use real observed desuckering counts instead.
       [feature_disabled] field_losses week_index=285 tag=Azul: no field-loss count reported for this active tag/week; assuming zero exogenous removal (the paper has no mechanistic flow to estimate this from).
       [feature_disabled] field_losses week_index=319 tag=Blanco: no field-loss count reported for this active tag/week; assuming zero exogenous removal (the paper has no mechanistic flow to estimate this from).
       [feature_disabled] field_losses week_index=331 tag=Negro: no field-loss count reported for this active tag/week; assuming zero exogenous removal (the paper has no mechanistic flow to e

,week_index,date,Nt,Nt_pre,Nt_post,Ht,FLt,St,sucker_output_pre,sucker_output_post,Dt,sucker_input_used,flowering_routed_to_unassigned,notice_count,pruned_population_this_week
356,512,2025-10-27,432916.969582,317885.381608,115031.587974,9034.011096,8060.448599,9290.290004,9234.572336,55.717668,0.0,9290.290004,0.0,0,0.439624
357,513,2025-11-03,431317.907529,316661.838801,114656.068728,8940.519246,7792.340736,6569.237049,6521.767351,47.469698,0.0,6569.237049,0.0,0,0.439119
358,514,2025-11-10,429849.907908,315763.197009,114086.710899,8844.357829,7540.722729,6642.560430,6607.540747,35.019683,0.0,6642.560430,0.0,0,0.479494
359,515,2025-11-17,431834.045540,316737.722669,115096.322871,7038.388028,7502.345013,8477.326605,8449.222926,28.103679,0.0,8477.326605,0.0,0,0.455932
360,516,2025-11-24,437633.082329,317878.910988,119754.171341,3291.151529,7651.257960,8792.905964,8763.307655,29.598308,0.0,8792.905964,0.0,0,0.459685


## 9. Mode C -- ribbon production + sucker selection

Mode B plus the farm's weekly desuckering totals (parous + nulliparous) as the pre-flowering
inflow; a week with no desuckering record falls back to the mechanistic prediction, with a notice
for that week. Skipped when the data directory has no `sucker_selection.csv`.

In [13]:
mode_c_engine, mode_c_output = None, None
if "Mode C" in MODES:
    mode_c_engine = build_engine("Mode C")
    mode_c_output = mode_c_engine.run(N_WEEKS)
    print("Mode C notices:")
    print_notices(mode_c_engine.notices)
    display(mode_c_output.tail())
else:
    print("no sucker_selection.csv in DATA_DIR -- Mode C skipped")

engines = {"Mode A": mode_a_engine, "Mode B": mode_b_engine, "Mode C": mode_c_engine}
outputs = {"Mode A": mode_a_output, "Mode B": mode_b_output, "Mode C": mode_c_output}

Mode C notices:
     2 x [fallback_estimate] sucker_selection
     4 x [feature_disabled] field_losses
       [feature_disabled] field_losses week_index=285 tag=Azul: no field-loss count reported for this active tag/week; assuming zero exogenous removal (the paper has no mechanistic flow to estimate this from).
       [feature_disabled] field_losses week_index=319 tag=Blanco: no field-loss count reported for this active tag/week; assuming zero exogenous removal (the paper has no mechanistic flow to estimate this from).
       [feature_disabled] field_losses week_index=331 tag=Negro: no field-loss count reported for this active tag/week; assuming zero exogenous removal (the paper has no mechanistic flow to estimate this from).
       [feature_disabled] field_losses week_index=497 tag=Rojo: no field-loss count reported for this active tag/week; assuming zero exogenous removal (the paper has no mechanistic flow to estimate this from).
       [fallback_estimate] sucker_selection week_index

,week_index,date,Nt,Nt_pre,Nt_post,Ht,FLt,St,sucker_output_pre,sucker_output_post,Dt,sucker_input_used,flowering_routed_to_unassigned,notice_count,pruned_population_this_week
356,512,2025-10-27,437360.325698,322328.737724,115031.587974,9034.011096,8897.737570,8375.149111,8308.230466,66.918645,0.0,9287.0,0.0,0,0.432175
357,513,2025-11-03,437076.130811,322420.062083,114656.068728,8940.519246,8738.228478,5867.518018,5807.462960,60.055058,0.0,8830.0,0.0,0,0.447163
358,514,2025-11-10,436094.214343,322007.503444,114086.710899,8844.357829,8451.081261,6194.000701,6150.802546,43.198154,0.0,8039.0,0.0,0,0.477378
359,515,2025-11-17,436336.020059,321239.697188,115096.322871,7038.388028,8221.310179,7322.819194,7292.218151,30.601043,0.0,7454.0,0.0,0,0.496076
360,516,2025-11-24,440148.601796,320394.430454,119754.171341,3291.151529,8114.773886,6981.344144,6952.173473,29.170671,0.0,7270.0,0.0,0,0.492847


In [14]:
# Notice subjects per mode, side by side
pd.DataFrame({mode: Counter(n.subject for n in engines[mode].notices) for mode in MODES}).fillna(0).astype(int)

,Mode A,Mode B,Mode C
ribbon_production,1,0,0
sucker_selection,1,1,2
field_losses,0,4,4


## 10. Full-history comparison -- Fig 2 / Fig 5 / Fig 6 / Table 4 style

The paper's diagnostics for every mode over the scoring window, joined to the observed cut by
`week_index`. Modes B/C are driven by the farm's own bagging, so their `Ht` should track the
observed cut closely; Mode A shows how far temperature alone gets from the same starting inventory.

In [15]:
MODE_COLORS = {"Mode A": "#F9A825", "Mode B": "#C62828", "Mode C": "#1565C0", "naive baseline": "#9E9E9E"}


def scored(mode: str) -> pd.DataFrame:
    """One row per scored week: date, simulated Ht, observed cut."""
    sim = outputs[mode].set_index("week_index").reindex(SCORED_WEEKS)
    return pd.DataFrame({"date": week_dates, "Ht": sim["Ht"], "obs_cut": observed_cut})


def r2(observed, predicted) -> float:
    observed, predicted = np.asarray(observed, dtype=float), np.asarray(predicted, dtype=float)
    return 1.0 - np.sum((observed - predicted) ** 2) / np.sum((observed - observed.mean()) ** 2)

### Fig 2 style -- harvest dynamics over time

Every mode starts from the farm's own inventory at the farm's own scale, so simulated `Ht` and the
observed cut share one bunch axis: magnitude is compared as well as timing.

In [16]:
def plot_full_history(mode: str) -> None:
    frame = scored(mode)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=frame["date"], y=frame["Ht"], name="Simulated (Ht)", line=dict(color="#1565C0")))
    fig.add_trace(go.Scatter(x=frame["date"], y=frame["obs_cut"], name="Observed (cut)", line=dict(color="#2E7D32")))
    fig.update_layout(
        title=f"{mode}: harvest dynamics (weeks {START_WEEK_INDEX}-{LAST_USABLE_WEEK})",
        xaxis_title="Date", yaxis_title="Bunches per week", template="plotly_white", legend=dict(orientation="h", y=1.15),
    )
    fig.show()


for mode in MODES:
    plot_full_history(mode)

### Fig 5 style -- weekly scatter and r²

The paper's own r²=0.9858 is computed on **1058 pooled weekly** `(simulated, observed)` pairs
across 7 validation fields -- not on a cumulative curve (a cumulative-sum scatter gives a
near-trivially high r² regardless of fit quality, since both series only increase). Reproduced
here on weekly pairs over the whole scoring window, for every mode. r² = 1 - SS_res / SS_tot
around the observed mean, so it is negative when the simulation is worse than a flat line at the
observed mean.

In [17]:
def plot_weekly_scatter(mode: str) -> float:
    frame = scored(mode)
    score = r2(frame["obs_cut"], frame["Ht"])
    fig = go.Figure()
    fig.update_layout(width=600, height=600)
    fig.add_trace(go.Scatter(x=frame["obs_cut"], y=frame["Ht"], mode="markers", name="Weekly pairs", marker=dict(color="#1565C0")))
    hi = max(frame["obs_cut"].max(), frame["Ht"].max())
    fig.add_trace(go.Scatter(x=[0, hi], y=[0, hi], mode="lines", line=dict(color="black", dash="dash"), name="y=x", showlegend=False))
    fig.update_layout(
        title=f"{mode}: simulated vs. observed weekly harvest (r\u00b2={score:.3f}, n={len(frame)})",
        xaxis_title="Observed (cut)", yaxis_title="Simulated (Ht)", template="plotly_white",
    )
    fig.show()
    return score


fig5_r2 = {mode: plot_weekly_scatter(mode) for mode in MODES}
for mode, score in fig5_r2.items():
    print(f"{mode}: weekly r\u00b2 = {score:.3f}")

Mode A: weekly r² = 0.899
Mode B: weekly r² = 0.998
Mode C: weekly r² = 0.998


### Fig 6 / Table 4 style -- harvesting-peak timing

Detect harvesting peaks independently in each series (`scipy.signal.find_peaks`, `distance=20`
weeks -- a floor justified by the model's own mechanics: harvest exhausts a cohort within 3 weeks
of threshold crossing, and flowering dispersion spreads one generation over a handful of weeks, so
anything narrower than ~20 weeks within a single series is sub-structure, not a distinct cycle),
then match observed and simulated peaks by solving an assignment problem (not greedy
nearest-neighbour, which can double-assign) with a data-driven tolerance. **Peak parameters are a
floor, not a target -- sanity-check the marked peaks below visually before trusting the table.**

In [18]:
def detect_and_match_peaks(sim: pd.Series, obs: pd.Series, *, distance: int = 20, prominence_frac: float = 0.10):
    sim_idx, _ = find_peaks(sim.to_numpy(), distance=distance, prominence=prominence_frac * sim.max())
    obs_idx, _ = find_peaks(obs.to_numpy(), distance=distance, prominence=prominence_frac * obs.max())
    sim_weeks, obs_weeks = sim.index.to_numpy()[sim_idx], obs.index.to_numpy()[obs_idx]

    if len(obs_weeks) == 0 or len(sim_weeks) == 0:
        return sim_weeks, obs_weeks, pd.DataFrame(columns=["observed_week", "simulated_week", "error_weeks"])

    tolerance = max(10.0, 0.5 * np.median(np.diff(np.sort(obs_weeks)))) if len(obs_weeks) > 1 else 10.0
    cost = np.abs(obs_weeks[:, None] - sim_weeks[None, :]).astype(float)
    row_ind, col_ind = linear_sum_assignment(cost)

    pairs = []
    for r, c in zip(row_ind, col_ind):
        if cost[r, c] <= tolerance:
            pairs.append({"observed_week": obs_weeks[r], "simulated_week": sim_weeks[c], "error_weeks": sim_weeks[c] - obs_weeks[r]})
    unmatched_obs = set(obs_weeks) - {p["observed_week"] for p in pairs}
    for w in sorted(unmatched_obs):
        pairs.append({"observed_week": w, "simulated_week": np.nan, "error_weeks": np.nan})
    matched = pd.DataFrame(pairs).sort_values("observed_week").reset_index(drop=True)
    return sim_weeks, obs_weeks, matched


def table4_stats(matched: pd.DataFrame, label: str) -> dict:
    err = matched["error_weeks"].dropna().astype(float)
    if err.empty:
        return {"label": label, "n": 0, "MD": np.nan, "MPE (=MAE)": np.nan, "MSEP": np.nan, "RMSEP": np.nan}
    return {
        "label": label, "n": len(err),
        "MD": err.mean(), "MPE (=MAE)": err.abs().mean(),
        "MSEP": (err ** 2).mean(), "RMSEP": np.sqrt((err ** 2).mean()),
    }


peak_matches: dict[str, pd.DataFrame] = {}
peak_counts: dict[str, tuple[int, int]] = {}
for mode in MODES:
    frame = scored(mode)
    sim_peak_weeks, obs_peak_weeks, matched = detect_and_match_peaks(frame["Ht"], frame["obs_cut"])
    peak_matches[mode] = matched
    peak_counts[mode] = (len(sim_peak_weeks), len(obs_peak_weeks))
    print(f"\n{mode}: {len(sim_peak_weeks)} simulated peaks, {len(obs_peak_weeks)} observed peaks detected, "
          f"{matched['simulated_week'].notna().sum()} matched")
    display(matched)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=frame.index, y=frame["Ht"], name="Simulated", line=dict(color="#1565C0")))
    fig.add_trace(go.Scatter(x=frame.index, y=frame["obs_cut"], name="Observed", line=dict(color="#2E7D32")))
    fig.add_trace(go.Scatter(x=sim_peak_weeks, y=frame.loc[sim_peak_weeks, "Ht"], mode="markers", name="Simulated peaks", marker=dict(symbol="x", size=10, color="#1565C0")))
    fig.add_trace(go.Scatter(x=obs_peak_weeks, y=frame.loc[obs_peak_weeks, "obs_cut"], mode="markers", name="Observed peaks", marker=dict(symbol="x", size=10, color="#2E7D32")))
    fig.update_layout(title=f"{mode}: detected harvesting peaks (sanity check before trusting Table 4)", xaxis_title="week_index", yaxis_title="Bunches per week", template="plotly_white", legend=dict(orientation="h", y=1.15))
    fig.show()


Mode A: 11 simulated peaks, 11 observed peaks detected, 9 matched


,observed_week,simulated_week,error_weeks
0,171,171.0,0.0
1,204,203.0,-1.0
2,226,226.0,0.0
3,272,272.0,0.0
4,302,296.0,-6.0
5,329,329.0,0.0
6,364,364.0,0.0
7,400,NaN,NaN
8,421,NaN,NaN
9,470,470.0,0.0



Mode B: 10 simulated peaks, 11 observed peaks detected, 10 matched


,observed_week,simulated_week,error_weeks
0,171,171.0,0.0
1,204,204.0,0.0
2,226,226.0,0.0
3,272,272.0,0.0
4,302,302.0,0.0
5,329,329.0,0.0
6,364,364.0,0.0
7,400,400.0,0.0
8,421,421.0,0.0
9,470,470.0,0.0



Mode C: 10 simulated peaks, 11 observed peaks detected, 10 matched


,observed_week,simulated_week,error_weeks
0,171,171.0,0.0
1,204,204.0,0.0
2,226,226.0,0.0
3,272,272.0,0.0
4,302,302.0,0.0
5,329,329.0,0.0
6,364,364.0,0.0
7,400,400.0,0.0
8,421,421.0,0.0
9,470,470.0,0.0


In [19]:
table4 = pd.DataFrame([table4_stats(peak_matches[mode], f"{mode}: all peaks") for mode in MODES])
print(
    "Caveat: with one farm's history (not the paper's 7 pooled fields), these are per-pair "
    "point estimates, not pooled statistics -- read n alongside every error column."
)
table4

Caveat: with one farm's history (not the paper's 7 pooled fields), these are per-pair point estimates, not pooled statistics -- read n alongside every error column.


,label,n,MD,MPE (=MAE),MSEP,RMSEP
0,Mode A: all peaks,9,-0.111111,1.444444,8.111111,2.848001
1,Mode B: all peaks,10,0.000000,0.000000,0.000000,0.000000
2,Mode C: all peaks,10,0.000000,0.000000,0.000000,0.000000


## 11. Naive baseline

A forecasting concept, not a full-history one -- "assume the next weeks look like the most recently
observed week, held flat." It is built on exactly the cutoffs the rolling sweep uses
(`SWEEP_CUTOFFS`, Section 12), so every series below is scored on the same (cutoff, target week)
pairs.

In [20]:
SWEEP_CUTOFFS = list(range(START_WEEK_INDEX, LAST_USABLE_WEEK - FORECAST_HORIZON + 1))


def scored_forecast(records: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame.from_records(records)
    df["target_date"] = df["target_week"].map(week_dates)
    df["observed_cut"] = df["target_week"].map(observed_cut)
    return df


def baseline_forecast(cutoffs, horizon: int = FORECAST_HORIZON) -> pd.DataFrame:
    return scored_forecast([
        {"cutoff_week": c, "horizon_step": h, "target_week": c + h, "predicted_Ht": observed_cut[c]}
        for c in cutoffs for h in range(1, horizon + 1)
    ])


baseline = baseline_forecast(SWEEP_CUTOFFS)
print(f"{len(SWEEP_CUTOFFS)} cutoffs, week_index {SWEEP_CUTOFFS[0]}..{SWEEP_CUTOFFS[-1]} "
      f"(targets up to {SWEEP_CUTOFFS[-1] + FORECAST_HORIZON} = LAST_USABLE_WEEK)")

349 cutoffs, week_index 156..504 (targets up to 516 = LAST_USABLE_WEEK)


## 12. Rolling 12-week-ahead forecast sweep

For each mode, one continuous engine is stepped through the scoring window, capturing
`snapshot_cohorts()` after every week's `step()`. A cutoff labelled `W` means "having consumed
week `W`'s data" -- the forecast then predicts weeks `W+1..W+12` from a **fresh, short-lived,
purely mechanistic** engine seeded from that exact snapshot. `rolling_sweep` builds only the
requested mode's engine. Mode A consumes no real data, so its forecast from any cutoff is just its
own static trajectory -- the check printed below confirms it.

In [21]:
def forecast_from_snapshot(snapshot: pd.DataFrame, cutoff_week: int, horizon: int) -> list[dict]:
    forecast_engine = SimbaPopEngine(
        parameters=params, weather=weather, initial_population=snapshot, start_week_index=cutoff_week + 1
    )
    out = forecast_engine.run(horizon)
    return [
        {"cutoff_week": cutoff_week, "horizon_step": w - cutoff_week, "target_week": w, "predicted_Ht": ht}
        for w, ht in zip(out["week_index"], out["Ht"])
    ]


def rolling_sweep(mode: str, horizon: int = FORECAST_HORIZON, stride: int = 1) -> pd.DataFrame:
    engine = build_engine(mode)  # only the requested mode's engine
    records = []
    for cutoff in range(START_WEEK_INDEX, LAST_USABLE_WEEK - horizon + 1, stride):
        while engine.current_week_index <= cutoff:  # advance the live engine through week `cutoff`
            engine.step()
        records.extend(forecast_from_snapshot(engine.snapshot_cohorts(), cutoff, horizon))
    df = scored_forecast(records)
    df.insert(0, "mode", mode)
    return df


forecasts: dict[str, pd.DataFrame] = {}
for mode in MODES:
    forecasts[mode] = rolling_sweep(mode)
    print(f"{mode}: {forecasts[mode]['cutoff_week'].nunique()} cutoffs swept, {len(forecasts[mode])} forecast-week rows")

mode_a_static = forecasts["Mode A"]["target_week"].map(mode_a_output.set_index("week_index")["Ht"])
print(f"Mode A: max |sweep forecast - static trajectory| = "
      f"{(forecasts['Mode A']['predicted_Ht'] - mode_a_static).abs().max():.3g} bunches")
forecasts[MODES[-1]].head(FORECAST_HORIZON)

Mode A: 349 cutoffs swept, 4188 forecast-week rows
Mode B: 349 cutoffs swept, 4188 forecast-week rows
Mode C: 349 cutoffs swept, 4188 forecast-week rows
Mode A: max |sweep forecast - static trajectory| = 0 bunches


,mode,cutoff_week,horizon_step,target_week,predicted_Ht,target_date,observed_cut
0,Mode C,156,1,157,5868.054459,2019-01-07,6057.0
1,Mode C,156,2,158,4061.099455,2019-01-14,3988.0
2,Mode C,156,3,159,10067.482225,2019-01-21,9932.0
3,Mode C,156,4,160,8087.531528,2019-01-28,7962.0
4,Mode C,156,5,161,7102.783240,2019-02-04,6841.0
5,Mode C,156,6,162,7219.525000,2019-02-11,7045.0
6,Mode C,156,7,163,7260.037332,2019-02-18,7059.0
7,Mode C,156,8,164,7174.395686,2019-02-25,7086.0
8,Mode C,156,9,165,7123.869470,2019-03-04,7143.0
9,Mode C,156,10,166,7390.407504,2019-03-11,7328.0


## 13. Rolling-sweep metrics: MAE/RMSE by horizon

Per-horizon-week (1..12) MAE/RMSE pooled over every cutoff, for each mode and the naive persistence
baseline from Section 11.

In [22]:
series = {**{mode: forecasts[mode] for mode in MODES}, "naive baseline": baseline}
per_horizon_rows = []
for name, df in series.items():
    error = df["predicted_Ht"] - df["observed_cut"]
    for h, e in error.groupby(df["horizon_step"]):
        per_horizon_rows.append({"series": name, "horizon_step": h, "MAE": e.abs().mean(), "RMSE": np.sqrt((e ** 2).mean())})
per_horizon = pd.DataFrame(per_horizon_rows)

fig = make_subplots(rows=1, cols=2, subplot_titles=("MAE", "RMSE"))
for name in series:
    sub = per_horizon[per_horizon["series"] == name].sort_values("horizon_step")
    dash = "dot" if name == "naive baseline" else "solid"
    for col, metric in ((1, "MAE"), (2, "RMSE")):
        fig.add_trace(
            go.Scatter(x=sub["horizon_step"], y=sub[metric], name=name, legendgroup=name, showlegend=(col == 1), line=dict(color=MODE_COLORS[name], dash=dash)),
            row=1, col=col,
        )
fig.update_xaxes(title_text="Horizon (weeks ahead)")
fig.update_yaxes(title_text="Bunches per week", row=1, col=1)
fig.update_layout(title="Forecast error by horizon week", template="plotly_white", legend=dict(orientation="h", y=1.15))
fig.show()

per_horizon.pivot(index="horizon_step", columns="series", values="MAE").round(0)

series,Mode A,Mode B,Mode C,naive baseline
horizon_step,,,,
1,354.0,56.0,56.0,782.0
2,357.0,57.0,57.0,1072.0
3,358.0,61.0,61.0,1298.0
4,361.0,67.0,67.0,1349.0
5,364.0,77.0,77.0,1388.0
6,365.0,86.0,86.0,1394.0
7,366.0,91.0,91.0,1429.0
8,365.0,95.0,95.0,1422.0
9,365.0,102.0,102.0,1392.0


### Signed forecast-error distributions

The signed error is `predicted_Ht - observed_cut`: positive values indicate over-prediction and negative values indicate under-prediction. Each forecast mode is shown in its own histogram.

In [23]:
fig = make_subplots(
    rows=len(MODES),
    cols=1,
    shared_xaxes=True,
    subplot_titles=[f"{mode}: signed forecast errors" for mode in MODES],
    vertical_spacing=0.08,
 )

for row, mode in enumerate(MODES, start=1):
    frame = forecasts[mode].copy()
    signed_error = frame["predicted_Ht"] - frame["observed_cut"]
    fig.add_trace(
        go.Histogram(
            x=signed_error,
            name=mode,
            marker_color=MODE_COLORS[mode],
            opacity=0.85,
            nbinsx=30,
            showlegend=False,
        ),
        row=row,
        col=1,
    )
    fig.add_vline(
        x=0,
        line_dash="dash",
        line_color="black",
        row=row,
        col=1,
    )
    fig.update_yaxes(title_text="Count", row=row, col=1)

fig.update_xaxes(title_text="Signed error: predicted_Ht - observed_cut (bunches/week)", row=len(MODES), col=1)
fig.update_layout(
    title="Signed forecast-error distributions by mode",
    template="plotly_white",
    height=max(300, 280 * len(MODES)),
    bargap=0.05,
    showlegend=False,
 )
fig.show()

signed_error_summary = pd.DataFrame(
    {
        mode: forecasts[mode]["predicted_Ht"] - forecasts[mode]["observed_cut"]
        for mode in MODES
    }
 ).agg(["mean", "median", "std", "min", "max"]).T
signed_error_summary

,mean,median,std,min,max
Mode A,80.248011,76.236607,458.733192,-1020.395024,1510.139134
Mode B,71.127248,69.556685,84.270816,-225.055970,483.050402
Mode C,72.027547,71.280986,82.592773,-188.945541,407.141324


## 14. Pooled forecast vs. observed, all cutoffs and horizons

In [24]:
for mode in MODES:
    df = forecasts[mode]
    score = r2(df["observed_cut"], df["predicted_Ht"])
    fig = go.Figure()
    fig.update_layout(width=600, height=600)
    fig.add_trace(go.Scatter(x=df["observed_cut"], y=df["predicted_Ht"], mode="markers", marker=dict(color="#1565C0", size=4, opacity=0.4), name="Forecasts"))
    hi = max(df["observed_cut"].max(), df["predicted_Ht"].max())
    fig.add_trace(go.Scatter(x=[0, hi], y=[0, hi], mode="lines", line=dict(color="black", dash="dash"), showlegend=False))
    fig.update_layout(title=f"{mode}: pooled forecast vs. observed (r\u00b2={score:.3f}, n={len(df)})", xaxis_title="Observed (cut)", yaxis_title="Predicted (Ht)", template="plotly_white")
    fig.show()

## 15. Fan chart

Adjacent cutoffs (stride 1) share 11 of 12 target weeks and near-identical cohort snapshots --
their errors are not independent samples, so hundreds of overlaid fans would be both unreadable and
misleading if read as independent. The band below aggregates, per target week, every forecast that
targeted it into a 10th-90th percentile envelope around the median. Six individual cutoffs' own
12-week fans, spread evenly over the swept cutoffs, are shown separately.

In [25]:
sample_cutoffs = [SWEEP_CUTOFFS[i] for i in np.linspace(0, len(SWEEP_CUTOFFS) - 1, 6).astype(int)]
for mode in MODES:
    df = forecasts[mode]
    band = df.groupby("target_week")["predicted_Ht"].quantile([0.1, 0.5, 0.9]).unstack()
    band.columns = ["p10", "p50", "p90"]
    band_dates = band.index.map(week_dates)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=band_dates, y=band["p90"], line=dict(width=0), showlegend=False))
    fig.add_trace(go.Scatter(x=band_dates, y=band["p10"], fill="tonexty", fillcolor="rgba(21,101,192,0.2)", line=dict(width=0), name="10th-90th pct of forecasts"))
    fig.add_trace(go.Scatter(x=band_dates, y=band["p50"], name="Median forecast", line=dict(color="#1565C0")))
    fig.add_trace(go.Scatter(x=week_dates, y=observed_cut, name="Observed", line=dict(color="#2E7D32")))
    fig.update_layout(title=f"{mode}: forecast fan (all cutoffs, aggregated)", xaxis_title="Date", yaxis_title="Bunches per week", template="plotly_white", legend=dict(orientation="h", y=1.15))
    fig.show()

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=week_dates, y=observed_cut, name="Observed", line=dict(color="#2E7D32", width=3)))
    for c in sample_cutoffs:
        sub = df[df["cutoff_week"] == c]
        fig.add_trace(go.Scatter(x=sub["target_date"], y=sub["predicted_Ht"], name=f"Forecast from week {c}", mode="lines"))
    fig.update_layout(title=f"{mode}: individual forecast fans (6 sample cutoffs)", xaxis_title="Date", yaxis_title="Bunches per week", template="plotly_white", legend=dict(orientation="h", y=1.15))
    fig.show()

## 16. Mass balance -- where every bagged bunch ends up

For the post-flowering chains of Modes B/C: cumulative harvested + cumulative field losses (`Xt`)
+ still on the plant = initial post-flowering stock + cumulative bagged. `Xt` is field losses only
-- rejected bunches are part of `cut` (harvested) and are not removed a second time. Any remainder
is post-flowering cohorts pruned below `population_epsilon` or past the
`max_age_weeks_reproductive` (20-week) cap.

> `Xt` and `bagged_input` exist only on `TagWeeklyRecord` (`engine.get_tag_output()`), not on
> `EngineWeeklyRecord` (`engine.get_output()`) -- see the [quick start](../quickstart.rst). The
> "still on the plant" band therefore comes from `get_tag_output()`'s own `Nt` (post-flowering
> chains only); `get_output()`'s `Nt` also includes the pre-flowering chain.

In [26]:
def mass_balance_totals(engine: SimbaPopEngine) -> pd.DataFrame:
    tag_output = engine.get_tag_output()
    totals = tag_output.groupby("week_index").agg(
        date=("date", "first"), Nt=("Nt", "sum"), Ht=("Ht", "sum"), Xt=("Xt", "sum"), bagged_input=("bagged_input", "sum"),
    )
    totals["cum_bagged"] = totals["bagged_input"].cumsum()
    totals["cum_Ht"] = totals["Ht"].cumsum()
    totals["cum_Xt"] = totals["Xt"].cumsum()
    totals["inflow"] = initial_post_stock + totals["cum_bagged"]
    totals["gap"] = totals["inflow"] - (totals["cum_Ht"] + totals["cum_Xt"] + totals["Nt"])
    return totals


mass_balance: dict[str, pd.DataFrame] = {}
for mode in [m for m in MODES if m != "Mode A"]:
    engine = engines[mode]
    totals = mass_balance_totals(engine)
    mass_balance[mode] = totals

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=totals["date"], y=totals["cum_Ht"], name="Cumulative harvested", stackgroup="one", line=dict(color="#C62828")))
    fig.add_trace(go.Scatter(x=totals["date"], y=totals["cum_Xt"], name="Cumulative field losses (Xt)", stackgroup="one", line=dict(color="#F9A825")))
    fig.add_trace(go.Scatter(x=totals["date"], y=totals["Nt"], name="Still on the plant (Nt)", stackgroup="one", line=dict(color="#2E7D32")))
    fig.add_trace(go.Scatter(x=totals["date"], y=totals["inflow"], name="Initial stock + cumulative bagged", line=dict(color="black", dash="dash")))
    fig.update_layout(title=f"{mode}: fate of every bagged bunch, cumulative", xaxis_title="Date", yaxis_title="Bunches", template="plotly_white", legend=dict(orientation="h", y=1.15))
    fig.show()

    last = totals.iloc[-1]
    print(f"{mode}: max relative mass-balance gap {(totals['gap'].abs() / totals['inflow']).max():.2e} "
          f"(final gap {last['gap']:,.1f} bunches)")
    print(f"{mode}: {initial_post_stock:,.0f} initial stock + {last['cum_bagged']:,.0f} bagged = "
          f"{last['cum_Ht']:,.0f} harvested ({last['cum_Ht'] / last['inflow']:.1%}) + "
          f"{last['cum_Xt']:,.0f} field losses ({last['cum_Xt'] / last['inflow']:.1%}) + "
          f"{last['Nt']:,.0f} still on the plant ({last['Nt'] / last['inflow']:.1%})")

    tag_output = engine.get_tag_output()
    unassigned_bagged = tag_output.loc[tag_output["tag"] == UNASSIGNED_TAG, "bagged_input"].sum()
    print(f"{mode}: {unassigned_bagged:,.0f} of the bagged total is mechanistic flowering routed to {UNASSIGNED_TAG} "
          "on weeks without any ribbon coverage\n")

Mode B: max relative mass-balance gap 7.03e-16 (final gap -0.0 bunches)
Mode B: 94,820 initial stock + 3,127,734 bagged = 3,049,900 harvested (94.6%) + 52,900 field losses (1.6%) + 119,754 still on the plant (3.7%)
Mode B: 0 of the bagged total is mechanistic flowering routed to __unassigned__ on weeks without any ribbon coverage



Mode C: max relative mass-balance gap 7.03e-16 (final gap -0.0 bunches)
Mode C: 94,820 initial stock + 3,127,734 bagged = 3,049,900 harvested (94.6%) + 52,900 field losses (1.6%) + 119,754 still on the plant (3.7%)
Mode C: 0 of the bagged total is mechanistic flowering routed to __unassigned__ on weeks without any ribbon coverage



## 17. One forecast, close up

A single example (the median swept cutoff): predicted vs. actual harvest for the 12 weeks
immediately following that cutoff, with the cutoff itself marked.

In [27]:
example_cutoff = int(np.median(SWEEP_CUTOFFS))
example_cutoff_date = pd.Timestamp(weather.calendar_date_for_week(example_cutoff))
for mode in MODES:
    example = forecasts[mode][forecasts[mode]["cutoff_week"] == example_cutoff].sort_values("target_week")
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=example["target_date"], y=example["observed_cut"], name="Observed (cut)", mode="lines+markers", line=dict(color="#2E7D32")))
    fig.add_trace(go.Scatter(x=example["target_date"], y=example["predicted_Ht"], name="Forecast (Ht)", mode="lines+markers", line=dict(color="#C62828", dash="dot")))
    fig.add_vline(x=example_cutoff_date, line_dash="dash", line_color="gray")
    fig.add_annotation(x=example_cutoff_date, y=1, yref="paper", text="cutoff", showarrow=False, yshift=10)
    fig.update_layout(title=f"{mode}: example 12-week-ahead forecast from cutoff week {example_cutoff}", xaxis_title="Date", yaxis_title="Bunches harvested", template="plotly_white", legend=dict(orientation="h", y=1.15))
    fig.show()

## 18. Across the whole sweep -- cumulative totals and RMSE over time

Left: predicted vs. observed 12-week-ahead *cumulative* total, one point per cutoff, against the
1:1 line. Right: forecast RMSE as the cutoff moves through the sweep, against a **second naive
baseline variant**: the trailing 4-week average of the observed cut, held flat.

> **Important:** this notebook uses two naive-baseline variants on purpose, not interchangeably.
> Section 11's "last observed value, held flat" is used everywhere else (Sections 13, 19, 20) for
> continuity. The chart below instead uses "trailing 4-week average, held flat," a smoother
> baseline arguably fairer for a noisy weekly harvest series -- kept as an explicit,
> separately-named alternative rather than silently swapped in.

In [28]:
def baseline_trailing_avg(cutoffs, window: int = 4, horizon: int = FORECAST_HORIZON) -> pd.DataFrame:
    records = []
    for c in cutoffs:
        trailing_weeks = range(max(FIRST_PRODUCTION_WEEK, c - window + 1), c + 1)
        trailing_value = observed_cut_all.reindex(trailing_weeks, fill_value=0.0).mean()
        records.extend(
            {"cutoff_week": c, "horizon_step": h, "target_week": c + h, "predicted_Ht": trailing_value}
            for h in range(1, horizon + 1)
        )
    return scored_forecast(records)


trailing = baseline_trailing_avg(SWEEP_CUTOFFS)
naive_rmse = np.sqrt(
    ((trailing["predicted_Ht"] - trailing["observed_cut"]) ** 2).groupby(trailing["cutoff_week"]).mean()
).rename("naive_rmse")

sweep_summary: dict[str, pd.DataFrame] = {}
for mode in MODES:
    df = forecasts[mode].assign(sq_error=lambda d: (d["predicted_Ht"] - d["observed_cut"]) ** 2)
    per_cutoff = df.groupby("cutoff_week").agg(
        predicted_total=("predicted_Ht", "sum"), observed_total=("observed_cut", "sum"), mse=("sq_error", "mean"),
    )
    per_cutoff["rmse"] = np.sqrt(per_cutoff["mse"])
    per_cutoff = per_cutoff.join(naive_rmse)
    per_cutoff["cutoff_date"] = per_cutoff.index.map(week_dates)
    sweep_summary[mode] = per_cutoff

    fig = make_subplots(rows=1, cols=2, subplot_titles=("12-week-ahead total: predicted vs. observed", "Forecast error by cutoff date"))
    fig.add_trace(go.Scatter(x=per_cutoff["observed_total"], y=per_cutoff["predicted_total"], mode="markers", marker=dict(color="#1565C0", size=6), name="Cutoffs"), row=1, col=1)
    lo = min(per_cutoff["observed_total"].min(), per_cutoff["predicted_total"].min())
    hi = max(per_cutoff["observed_total"].max(), per_cutoff["predicted_total"].max())
    fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode="lines", line=dict(color="gray", dash="dash"), name="1:1", showlegend=False), row=1, col=1)
    fig.add_trace(go.Scatter(x=per_cutoff["cutoff_date"], y=per_cutoff["rmse"], name="Model RMSE", line=dict(color="#C62828")), row=1, col=2)
    fig.add_trace(go.Scatter(x=per_cutoff["cutoff_date"], y=per_cutoff["naive_rmse"], name="Naive baseline (trailing 4-week avg) RMSE", line=dict(color="#9E9E9E", dash="dot")), row=1, col=2)
    fig.update_xaxes(title_text="Observed 12-week total", row=1, col=1)
    fig.update_yaxes(title_text="Predicted 12-week total", row=1, col=1)
    fig.update_xaxes(title_text="Cutoff date", row=1, col=2)
    fig.update_yaxes(title_text="RMSE (bunches/week)", row=1, col=2)
    fig.update_layout(title=f"{mode}: sweep summary", template="plotly_white", height=450, legend=dict(orientation="h", y=1.2))
    fig.show()

    print(f"{mode}: model beats the trailing-average naive baseline in "
          f"{(per_cutoff['rmse'] < per_cutoff['naive_rmse']).sum()} of {len(per_cutoff)} cutoffs")

Mode A: model beats the trailing-average naive baseline in 339 of 349 cutoffs


Mode B: model beats the trailing-average naive baseline in 349 of 349 cutoffs


Mode C: model beats the trailing-average naive baseline in 349 of 349 cutoffs


## 19. Mode B vs. Mode C -- where the desuckering records matter

**Why 12-week harvest forecasts cannot tell them apart.** With complete real bagging, Modes B and C
seed their post-flowering chains identically. A bunch is cut roughly 12 weeks after it is bagged
(Tixier's 900 degree-day harvest threshold, then 20% / 62.5% / 17.5% over three weeks), so for
horizons up to ~11 weeks every bunch in the forecast window was already bagged -- and recorded --
by the cutoff, and both modes forecast exactly the same harvest by construction. Desuckering only
changes the pre-flowering chain, i.e. how many plants flower, and are bagged, in the future. Its
effect can therefore only show up in (a) the model's own mechanistic flowering, (b) the vegetative
stock `Nt_pre`, and (c) forecasts long enough to include bunches not yet bagged at the cutoff.

In [29]:
if "Mode C" not in MODES:
    print("no sucker_selection.csv in DATA_DIR -- Section 19 (Mode B vs. Mode C) skipped")
else:
    paired = forecasts["Mode B"].merge(
        forecasts["Mode C"], on=["cutoff_week", "horizon_step"], suffixes=("_B", "_C"), validate="one_to_one"
    )
    paired["abs_diff"] = (paired["predicted_Ht_B"] - paired["predicted_Ht_C"]).abs()
    identity = paired.groupby("horizon_step").agg(max_abs_diff_B_vs_C=("abs_diff", "max"))
    identity["MAE Mode B"] = per_horizon[per_horizon["series"] == "Mode B"].set_index("horizon_step")["MAE"]
    identity["MAE Mode C"] = per_horizon[per_horizon["series"] == "Mode C"].set_index("horizon_step")["MAE"]
    identical = identity.index[identity["max_abs_diff_B_vs_C"] < 1e-6]
    print(f"Mode B and Mode C 12-week forecasts are identical (to 1e-6 bunches) for horizons {list(identical)}")
    display(identity.round(3))

Mode B and Mode C 12-week forecasts are identical (to 1e-6 bunches) for horizons [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]


,max_abs_diff_B_vs_C,MAE Mode B,MAE Mode C
horizon_step,,,
1,0.000,56.283,56.283
2,0.000,57.214,57.214
3,0.000,60.697,60.697
4,0.000,66.700,66.700
5,0.000,77.440,77.440
6,0.000,86.069,86.069
7,0.000,91.205,91.205
8,0.000,95.117,95.117
9,0.000,101.803,101.803


### (a) Mechanistic flowering vs. recorded bagging

`FLt` is the model's own flowering prediction from the pre-flowering chain, always computed even
when real bagging drives the post-flowering chains. Mode C's pre-flowering chain is fed the farm's
real follower selections, so its `FLt` should track the recorded bagging; Mode B assumes Tixier's
selection curve and drifts in cycle timing when the farm desuckers differently. The synthetic
farms follow their own practice, py-simba-pop-tools' `GUARIN_OCHOA_2012_SUCKER_SELECTION` (Guarin & Ochoa 2012,
Table 2): a slightly lower threshold but a peak 5.5 rather than 3.25 weeks after it and a wider
spread, i.e. followers selected later and over more weeks than Tixier's calibrated curve -- both
curves are printed below, together with the share of followers selected after their mother had
already flowered. Mode A's `FLt` equals Mode B's: both use the same mechanistic pre-flowering
inflow.

In [30]:
if "Mode C" not in MODES:
    print("no sucker_selection.csv in DATA_DIR -- skipped")
else:
    from py_simba_pop_tools.calibration import GUARIN_OCHOA_2012_SUCKER_SELECTION as FARM_PRACTICE

    curves = pd.DataFrame(
        {
            "Tixier default (Modes A/B fallback)": {k: params.sucker_selection[k] for k in FARM_PRACTICE},
            "synthetic farms' practice (FARM_PRACTICE)": FARM_PRACTICE,
        }
    ).T
    display(curves)

    fl = {mode: outputs[mode].set_index("week_index")["FLt"].reindex(SCORED_WEEKS) for mode in MODES}
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=week_dates, y=observed_bagged, name="Recorded bagging", line=dict(color="#2E7D32", width=3)))
    for mode in ("Mode B", "Mode C"):
        fig.add_trace(go.Scatter(x=week_dates, y=fl[mode], name=f"{mode} FLt", line=dict(color=MODE_COLORS[mode], dash="dot" if mode == "Mode C" else "solid")))
    fig.update_layout(title="Mechanistic flowering (FLt) vs. recorded weekly bagging", xaxis_title="Date", yaxis_title="Plants flowering / bunches bagged per week", template="plotly_white", legend=dict(orientation="h", y=1.15))
    fig.show()

    for mode in ("Mode B", "Mode C"):
        print(f"{mode}: r\u00b2(FLt vs. recorded bagging) = {r2(observed_bagged, fl[mode]):.3f}")
    print(f"Mode A vs. Mode B: max |FLt difference| = {(fl['Mode A'] - fl['Mode B']).abs().max():.3g}")

    sel = sel_result.data
    sel = sel[sel["date"].map(weather.week_index_for_date).between(START_WEEK_INDEX, LAST_USABLE_WEEK)]
    parous_mech = mode_b_output["sucker_output_post"].sum() / mode_b_output["St"].sum()
    print(f"share of followers selected for already-flowered (parous) mothers: recorded {sel['parous'].sum() / sel['total'].sum():.1%}, "
          f"Mode B's mechanistic prediction {parous_mech:.1%}")

,threshold_dd,amplitude,peak_weeks,spread
Tixier default (Modes A/B fallback),2180.0,1.000,3.25,0.310
synthetic farms' practice (FARM_PRACTICE),2080.0,0.171,5.50,0.379


Mode B: r²(FLt vs. recorded bagging) = 0.435
Mode C: r²(FLt vs. recorded bagging) = 1.000
Mode A vs. Mode B: max |FLt difference| = 0
share of followers selected for already-flowered (parous) mothers: recorded 9.5%, Mode B's mechanistic prediction 0.7%


### (b) The vegetative stock `Nt_pre`

Modes A and B share the same mechanistic pre-flowering inflow, so their `Nt_pre` lines coincide;
Mode C's follows the recorded selections.

In [31]:
if "Mode C" not in MODES:
    print("no sucker_selection.csv in DATA_DIR -- skipped")
else:
    fig = go.Figure()
    for mode in MODES:
        out = outputs[mode]
        fig.add_trace(go.Scatter(x=out["date"], y=out["Nt_pre"], name=f"{mode} Nt_pre", line=dict(color=MODE_COLORS[mode], dash="dash" if mode == "Mode A" else "solid")))
    fig.update_layout(title="Pre-flowering (vegetative) stock Nt_pre by mode", xaxis_title="Date", yaxis_title="Plants", template="plotly_white", legend=dict(orientation="h", y=1.15))
    fig.show()
    for mode in MODES:
        nt_pre = outputs[mode]["Nt_pre"]
        print(f"{mode}: Nt_pre {nt_pre.iloc[0]:,.0f} after the first week, {nt_pre.iloc[-1]:,.0f} at the end "
              f"(range {nt_pre.min():,.0f}..{nt_pre.max():,.0f})")

Mode A: Nt_pre 323,861 after the first week, 317,879 at the end (range 307,440..334,834)
Mode B: Nt_pre 323,861 after the first week, 317,879 at the end (range 307,440..334,834)
Mode C: Nt_pre 322,795 after the first week, 320,394 at the end (range 317,800..329,185)


### (c) A 26-week horizon

A second rolling sweep, 26 weeks ahead with a stride of 4 cutoffs (to keep it quick), for Modes B
and C plus the naive baseline on the same cutoffs. Horizons beyond ~12 weeks include bunches not yet
bagged at the cutoff, so they depend on the pre-flowering chain -- where Mode C's desuckering data
enters. `FORECAST_HORIZON` stays 12 for the main sweep above.

In [32]:
if "Mode C" not in MODES:
    print("no sucker_selection.csv in DATA_DIR -- skipped")
else:
    long_cutoffs = list(range(START_WEEK_INDEX, LAST_USABLE_WEEK - LONG_HORIZON + 1, LONG_STRIDE))
    long_series = {mode: rolling_sweep(mode, horizon=LONG_HORIZON, stride=LONG_STRIDE) for mode in ("Mode B", "Mode C")}
    long_series["naive baseline"] = baseline_forecast(long_cutoffs, LONG_HORIZON)
    long_mae = pd.DataFrame(
        {name: (df["predicted_Ht"] - df["observed_cut"]).abs().groupby(df["horizon_step"]).mean() for name, df in long_series.items()}
    )

    fig = go.Figure()
    for name in long_mae.columns:
        fig.add_trace(go.Scatter(x=long_mae.index, y=long_mae[name], name=name, line=dict(color=MODE_COLORS[name], dash="dot" if name == "naive baseline" else "solid")))
    fig.add_vline(x=FORECAST_HORIZON, line_dash="dash", line_color="gray")
    fig.update_layout(title=f"{LONG_HORIZON}-week-ahead forecast MAE by horizon ({len(long_cutoffs)} cutoffs, stride {LONG_STRIDE})", xaxis_title="Horizon (weeks ahead)", yaxis_title="MAE (bunches/week)", template="plotly_white", legend=dict(orientation="h", y=1.15))
    fig.show()

    print("mean MAE, horizons 1-11 vs. 13-26:")
    for name in long_mae.columns:
        print(f"  {name}: {long_mae.loc[1:11, name].mean():,.0f} vs. {long_mae.loc[13:, name].mean():,.0f}")
    display(long_mae.loc[[1, 6, 11, 12, 13, 16, 20, 26]].round(0))

mean MAE, horizons 1-11 vs. 13-26:
  Mode B: 84 vs. 354
  Mode C: 84 vs. 146
  naive baseline: 1,274 vs. 1,443


,Mode B,Mode C,naive baseline
horizon_step,,,
1,58.0,58.0,805.0
6,82.0,82.0,1474.0
11,120.0,120.0,1389.0
12,143.0,144.0,1460.0
13,232.0,150.0,1326.0
16,379.0,150.0,1478.0
20,388.0,148.0,1538.0
26,346.0,141.0,1550.0


## 20. Scorecard -- every mode and the naive baseline together

Full-history fit (Section 10) and rolling 12-week forecast error (Section 13) side by side; skill
= 1 - MAE / MAE of the naive baseline at the same horizon (positive = better than holding the last
observed week flat).

In [33]:
mae_by = per_horizon.pivot(index="horizon_step", columns="series", values="MAE")
scorecard_rows = []
for name in [*MODES, "naive baseline"]:
    row = {"series": name}
    if name in MODES:
        stats = table4_stats(peak_matches[name], name)
        row.update({
            "Fig 5 r\u00b2 (weekly)": fig5_r2[name],
            "peaks matched / observed": f"{stats['n']} / {peak_counts[name][1]}",
            "Table 4 RMSEP (weeks)": stats["RMSEP"],
        })
    row.update({
        "MAE h=1": mae_by.loc[1, name],
        f"MAE h={FORECAST_HORIZON}": mae_by.loc[FORECAST_HORIZON, name],
        f"mean MAE h=1-{FORECAST_HORIZON}": mae_by[name].mean(),
        "skill h=1": 1 - mae_by.loc[1, name] / mae_by.loc[1, "naive baseline"],
        f"skill h={FORECAST_HORIZON}": 1 - mae_by.loc[FORECAST_HORIZON, name] / mae_by.loc[FORECAST_HORIZON, "naive baseline"],
    })
    scorecard_rows.append(row)
scorecard = pd.DataFrame(scorecard_rows).set_index("series")
scorecard.round(3)

,Fig 5 r² (weekly),peaks matched / observed,Table 4 RMSEP (weeks),MAE h=1,MAE h=12,mean MAE h=1-12,skill h=1,skill h=12
series,,,,,,,,
Mode A,0.899,9 / 11,2.848,354.478,365.719,362.720,0.547,0.736
Mode B,0.998,10 / 11,0.000,56.283,137.099,88.978,0.928,0.901
Mode C,0.998,10 / 11,0.000,56.283,137.700,89.028,0.928,0.901
naive baseline,NaN,NaN,NaN,781.679,1387.006,1307.365,0.000,0.000


## Takeaways (default `data/synthetic` run)

- **One engine, three ways to feed it.** Modes A, B and C are the same `SimbaPopEngine`
  constructor with more of its optional real-data arguments filled in (`build_engine`), never a
  different code path.
- **Mode A stays at the farm's scale without any carrying capacity.** From the census and
  temperature alone, `Nt` stayed within 408,279-443,383 plants over 361 weeks (census: 417,296)
  and the simulated harvest averaged 445,026 bunches/yr against 440,751 observed. Weekly r² was
  0.899, and 9 of 11 observed harvesting peaks were matched (RMSEP 2.8 weeks). Its 12-week
  "forecast" is its own static trajectory (MAE 354-366 bunches/week at every horizon), still well
  below the naive baseline's 782-1,429. The synthetic farm shares Mode A's biology and differs
  mainly in its desuckering practice (plus counting noise and field losses), so this is an
  optimistic case for Mode A.
- **Real bagging makes the harvest nearly exact (Modes B/C).** Weekly r² 0.998; 10 of 11 observed
  peaks matched with zero timing error (the last observed peak, week 499, has no detected
  simulated counterpart although the curves overlap there -- a peak-detection threshold effect);
  12-week MAE 56 bunches/week at h=1 and 137-138 at h=12, against 782 and 1,387 for the naive
  baseline; both beat the trailing-average baseline at all 349 cutoffs. The post-flowering mass
  balance closes to a relative gap of ~7e-16.
- **Mode B vs. Mode C.** Their 12-week forecasts are identical for horizons 1-11 by construction
  and differ only at h=12 (MAE 137.1 vs. 137.7). The desuckering records matter where the
  mechanism says they should: Mode C's mechanistic flowering reproduces the recorded bagging (r²
  1.000 -- by construction here, since the synthetic farm is this model plus its own practice)
  while Mode B's drifts (r² 0.435); 9.5% of the recorded selections were for already-flowered
  mothers vs. 0.7% under Tixier's curve; and 13-26 weeks ahead Mode C's mean MAE is 146
  bunches/week vs. 354 for Mode B (naive baseline: 1,443).
- **Every mode is scored against a naive persistence baseline** -- in two explicit variants
  (Section 11's last value held flat, Section 18's trailing 4-week average).
- **Before trusting a real farm's run**, check Section 5's bagging-coverage scan and re-read every
  printed number: rerunning with `SIMBA_POP_DATA_DIR` changes all of them, and a farm without
  `sucker_selection.csv` skips Mode C and Section 19.